%% [markdown]
# 03c — NHS ODS GP Surgery Locations Audit

In [1]:
# %%
import os, pandas as pd, numpy as np, re
from pathlib import Path
from pyproj import Transformer

In [2]:
ROOT = Path("/Users/souravamseekarmarti/Projects/aequitas")
DATA = ROOT / "data"

In [3]:
# %%
# epraccur.csv has no header — assign column names
gps = pd.read_csv(DATA / "raw/poi/epraccur.csv", header=None,
                  names=['OrgId','Name','PostCode','LastChangeDate'])
print(f"Rows: {len(gps)}, Cols: {gps.columns.tolist()}")
print(gps.head(3))

Rows: 12214, Cols: ['OrgId', 'Name', 'PostCode', 'LastChangeDate']
    OrgId                        Name  PostCode  LastChangeDate
0   OrgId                        Name  PostCode  LastChangeDate
1  A81002  QUEENS PARK MEDICAL CENTRE  TS18 2AW      2024-06-11
2  A81004       ACKLAM MEDICAL CENTRE   TS5 8SB      2023-08-22


In [4]:
# %%
assert len(gps) >= 12000, f"FAIL: expected 12213, got {len(gps)}"
print(f"CHECK PASS: {len(gps)} rows")
assert gps['OrgId'].nunique() == len(gps), "FAIL: duplicate OrgIds"
null_check = gps[['OrgId','Name','PostCode']].isnull().sum()
assert null_check.sum() == 0, f"FAIL: nulls: {null_check[null_check>0]}"
print("CHECK PASS: no nulls, no duplicates")

CHECK PASS: 12214 rows
CHECK PASS: no nulls, no duplicates


In [5]:
# %%
def standardise_postcode(pc):
    if pd.isna(pc): return None
    pc = re.sub(r'\s+', '', str(pc).strip().upper())
    return pc[:-3] + ' ' + pc[-3:] if len(pc) > 3 else pc

In [6]:
gps['postcode_std'] = gps['PostCode'].apply(standardise_postcode)
print(f"Sample: {gps['postcode_std'].head(3).tolist()}")

Sample: ['POSTC ODE', 'TS18 2AW', 'TS5 8SB']


In [7]:
# %%
lookup = pd.read_parquet(DATA / "audit/postcode_lookup.parquet")
geocoded = gps.merge(lookup.rename(columns={'Postcode': 'postcode_std'}), on='postcode_std', how='left')
matched = geocoded['Easting'].notna().sum()
match_rate = matched / len(geocoded) * 100
print(f"Match rate: {match_rate:.1f}% ({matched}/{len(geocoded)})")
assert match_rate >= 95.0, f"FAIL: match rate {match_rate:.1f}% below 95%"
print("CHECK PASS: match rate >= 95%")

Match rate: 98.7% (12059/12214)
CHECK PASS: match rate >= 95%


In [8]:
# %%
t = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=True)
has_coords = geocoded['Easting'].notna()
lons, lats = t.transform(geocoded.loc[has_coords, 'Easting'].values, geocoded.loc[has_coords, 'Northing'].values)
geocoded.loc[has_coords, 'longitude'] = lons
geocoded.loc[has_coords, 'latitude'] = lats
print(f"Lat: {geocoded['latitude'].min():.3f} to {geocoded['latitude'].max():.3f}")
print(f"Lon: {geocoded['longitude'].min():.3f} to {geocoded['longitude'].max():.3f}")
assert geocoded['latitude'].dropna().between(49.8, 55.8).all(), "FAIL: lat out of bounds"
assert geocoded['longitude'].dropna().between(-6.4, 1.8).all(), "FAIL: lon out of bounds"
print("CHECK PASS: all coords within England bounds")

Lat: 49.913 to 55.596
Lon: -6.309 to 1.755
CHECK PASS: all coords within England bounds


In [9]:
# %%
out = geocoded[['OrgId','Name','PostCode','postcode_std','Easting','Northing','latitude','longitude']].copy()
out.to_parquet(DATA / "audit/gp_surgeries_geocoded.parquet", index=False)
print(f"Saved: {len(out)} rows, {out['latitude'].notna().sum()} geocoded ({match_rate:.1f}%)")
print("03c_gp_surgeries COMPLETE")

Saved: 12214 rows, 12059 geocoded (98.7%)
03c_gp_surgeries COMPLETE
